# 08_2 DQA Scene Phase2 Head-Protected Policy

08 の phase1 checkpoint を seed にして、phase2 だけを追加学習する比較実験です。target client では detector head を守るため `train_scope=backbone` にし、low/mid pseudoGT は positive objectness ではなく ignore として扱います。


## 1. Paths

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import socket
import subprocess
import sys
from collections import deque
from pathlib import Path
from typing import Optional

import pandas as pd


def find_repo_root(start: Optional[Path] = None) -> Path:
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    required = (
        "dynamic_quality_aware_classwise_aggregation/run_dqa_cwa_fedsto_scene_v2_phase2_head_protected_policy.py",
        "dynamic_quality_aware_classwise_aggregation/evaluate_scene_protocol.py",
        "navigating_data_heterogeneity/setup_fedsto_scene_reproduction.py",
    )
    for base in (start, *start.parents):
        for candidate in (base, base / "Object_Detection"):
            if all((candidate / marker).exists() for marker in required):
                return candidate.resolve()
    raise FileNotFoundError("Could not locate /app/Object_Detection")


REPO_ROOT = find_repo_root()
DQA_ROOT = REPO_ROOT / "dynamic_quality_aware_classwise_aggregation"
RUN_SCRIPT = DQA_ROOT / "run_dqa_cwa_fedsto_scene_v2_phase2_head_protected_policy.py"
EVAL_SCRIPT = DQA_ROOT / "evaluate_scene_protocol.py"
POLICY_MODEL = DQA_ROOT / "threshold_policy_model" / "artifacts" / "dqa05_threshold_policy.joblib"

SOURCE_WORK_ROOT = DQA_ROOT / "efficientteacher_dqa08_scene_tri_stage_policy_8h"
WORK_ROOT = DQA_ROOT / "efficientteacher_dqa08_2_scene_phase2_head_protected"
STATS_ROOT = DQA_ROOT / "stats_dqa08_2_scene_phase2_head_protected"
RUNNER_LOG = DQA_ROOT / "dqa08_2_scene_phase2_head_protected_runner.out"
TRAIN_LOG = DQA_ROOT / "dqa08_2_scene_phase2_head_protected_train.log"
PID_PATH = DQA_ROOT / "dqa08_2_scene_phase2_head_protected_runner.pid"
THRESHOLD_LOG = STATS_ROOT / "head_protected_policy_schedule.jsonl"

preferred_python = Path("/root/micromamba/envs/al_yolov8/bin/python")
PYTHON_BIN = preferred_python if preferred_python.exists() else Path(sys.executable)

print("repo_root:", REPO_ROOT)
print("source_08_workspace:", SOURCE_WORK_ROOT)
print("workspace:", WORK_ROOT)
print("stats_root:", STATS_ROOT)
print("python:", PYTHON_BIN)
print("policy_model:", POLICY_MODEL)
print("threshold_log:", THRESHOLD_LOG)


## 2. Experiment Settings

In [ ]:
# 08_2 starts from an existing 08 phase1 checkpoint and runs phase2 only.
# Round 12 is the strict comparison seed because 08 phase2 also started there.
# For a best-phase1 ablation, change SOURCE_PHASE1_ROUND to 3.
SOURCE_PHASE1_ROUND = 12
SOURCE_PHASE2_BASELINE_ROUND = 24
FORCE_SEED_PHASE1 = False

WARMUP_EPOCHS = 0
PHASE1_ROUNDS = 0
PHASE2_ROUNDS = 24
DQA_START_PHASE = 2

BATCH_SIZE = 160
WORKERS = 8
REQUESTED_GPUS = 2
MIN_FREE_GIB = 8

# Head-protected phase2 profile.
CLIENT_TRAIN_SCOPE = "backbone"
SERVER_TRAIN_SCOPE = "all"
UNCERTAIN_IGNORE = True
CLIENT_LR0 = 3e-4
SERVER_LR0 = 1e-3
SERVER_ORTHOGONAL_WEIGHT = 1e-4

# Tri-stage learned policy is still used to decide low/mid/high gates.
SSOD_PROFILE = "tri_stage_head_protected_policy"
ADAPT_START_ROUND = 3
POLICY_HORIZON_ROUNDS = PHASE2_ROUNDS
MIN_LOW = 0.38
MAX_LOW = 0.48
MIN_MID = 0.58
MAX_MID = 0.72
MIN_HIGH = 0.84
MAX_HIGH = 0.92
LOW_SHIFT = 0.02
HIGH_SHIFT = 0.05
MID_GAP = 0.20
HIGH_MID_GAP = 0.18
MIN_HIGH_GAP = 0.16
MAX_NMS = 0.45
TEACHER_MIN = 0.28
RARE_COUNT = 250
RARE_MAX_LOW = 0.43
RARE_MAX_MID = 0.66
RARE_MAX_HIGH = 0.89
LOW_STEP_LIMIT = 0.02
MID_STEP_LIMIT = 0.025
HIGH_STEP_LIMIT = 0.035
NMS_STEP_LIMIT = 0.02
LOW_MID_OBJ_WEIGHT = 0.45
MID_HIGH_OBJ_WEIGHT = 1.0

APPEND_TRAIN_LOG = False
RUN_TRAINING = True
RUN_IN_BACKGROUND = False
STREAM_TRAIN_OUTPUT = True

try:
    import torch

    AVAILABLE_CUDA_GPUS = torch.cuda.device_count()
except Exception as exc:
    AVAILABLE_CUDA_GPUS = 0
    print("Could not inspect CUDA devices:", exc)

GPUS = min(REQUESTED_GPUS, AVAILABLE_CUDA_GPUS) if AVAILABLE_CUDA_GPUS else 1
if GPUS != REQUESTED_GPUS:
    print(f"Requested {REQUESTED_GPUS} GPU(s), visible={AVAILABLE_CUDA_GPUS}; using GPUS={GPUS}")


def find_free_port(preferred: int) -> int:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        try:
            sock.bind(("127.0.0.1", preferred))
            return preferred
        except OSError:
            pass
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind(("127.0.0.1", 0))
        return int(sock.getsockname()[1])


MASTER_PORT = find_free_port(29575)

os.environ["DQA08_2_SOURCE_WORK_ROOT"] = str(SOURCE_WORK_ROOT)
os.environ["DQA08_2_SOURCE_PHASE1_ROUND"] = str(SOURCE_PHASE1_ROUND)
os.environ["DQA08_2_FORCE_SEED"] = "1" if FORCE_SEED_PHASE1 else "0"
os.environ["DQA08_2_CLIENT_TRAIN_SCOPE"] = CLIENT_TRAIN_SCOPE
os.environ["DQA08_2_SERVER_TRAIN_SCOPE"] = SERVER_TRAIN_SCOPE
os.environ["DQA08_2_UNCERTAIN_IGNORE"] = "1" if UNCERTAIN_IGNORE else "0"
os.environ["DQA08_2_CLIENT_LR0"] = str(CLIENT_LR0)
os.environ["DQA08_2_SERVER_ORTHOGONAL_WEIGHT"] = str(SERVER_ORTHOGONAL_WEIGHT)

os.environ["DQA08_SSOD_PROFILE"] = SSOD_PROFILE
os.environ["DQA08_POLICY_MODEL"] = str(POLICY_MODEL)
os.environ["DQA08_POLICY_HORIZON_ROUNDS"] = str(POLICY_HORIZON_ROUNDS)
os.environ["DQA08_CLIENT_LR0"] = str(CLIENT_LR0)
os.environ["DQA08_SERVER_LR0"] = str(SERVER_LR0)
os.environ["DQA08_ADAPT_START_ROUND"] = str(ADAPT_START_ROUND)
os.environ["DQA08_MIN_LOW"] = str(MIN_LOW)
os.environ["DQA08_MAX_LOW"] = str(MAX_LOW)
os.environ["DQA08_MIN_MID"] = str(MIN_MID)
os.environ["DQA08_MAX_MID"] = str(MAX_MID)
os.environ["DQA08_MIN_HIGH"] = str(MIN_HIGH)
os.environ["DQA08_MAX_HIGH"] = str(MAX_HIGH)
os.environ["DQA08_LOW_SHIFT"] = str(LOW_SHIFT)
os.environ["DQA08_HIGH_SHIFT"] = str(HIGH_SHIFT)
os.environ["DQA08_MID_GAP"] = str(MID_GAP)
os.environ["DQA08_HIGH_MID_GAP"] = str(HIGH_MID_GAP)
os.environ["DQA08_MIN_HIGH_GAP"] = str(MIN_HIGH_GAP)
os.environ["DQA08_MAX_NMS"] = str(MAX_NMS)
os.environ["DQA08_TEACHER_MIN"] = str(TEACHER_MIN)
os.environ["DQA08_RARE_COUNT"] = str(RARE_COUNT)
os.environ["DQA08_RARE_MAX_LOW"] = str(RARE_MAX_LOW)
os.environ["DQA08_RARE_MAX_MID"] = str(RARE_MAX_MID)
os.environ["DQA08_RARE_MAX_HIGH"] = str(RARE_MAX_HIGH)
os.environ["DQA08_LOW_STEP_LIMIT"] = str(LOW_STEP_LIMIT)
os.environ["DQA08_MID_STEP_LIMIT"] = str(MID_STEP_LIMIT)
os.environ["DQA08_HIGH_STEP_LIMIT"] = str(HIGH_STEP_LIMIT)
os.environ["DQA08_NMS_STEP_LIMIT"] = str(NMS_STEP_LIMIT)
os.environ["DQA08_LOW_MID_OBJ_WEIGHT"] = str(LOW_MID_OBJ_WEIGHT)
os.environ["DQA08_MID_HIGH_OBJ_WEIGHT"] = str(MID_HIGH_OBJ_WEIGHT)
os.environ["DQA08_THRESHOLD_LOG"] = str(THRESHOLD_LOG)

{
    "source_phase1_round": SOURCE_PHASE1_ROUND,
    "phase1_rounds": PHASE1_ROUNDS,
    "phase2_rounds": PHASE2_ROUNDS,
    "dqa_start_phase": DQA_START_PHASE,
    "client_train_scope": CLIENT_TRAIN_SCOPE,
    "uncertain_ignore": UNCERTAIN_IGNORE,
    "policy_model_exists": POLICY_MODEL.exists(),
    "client_lr0": CLIENT_LR0,
    "server_lr0": SERVER_LR0,
    "adapt_start_round": ADAPT_START_ROUND,
    "low_range": (MIN_LOW, MAX_LOW),
    "mid_range": (MIN_MID, MAX_MID),
    "high_range": (MIN_HIGH, MAX_HIGH),
    "batch_size": BATCH_SIZE,
    "workers": WORKERS,
    "gpus": GPUS,
    "master_port": MASTER_PORT,
}


## 3. Build Lists and Seed Phase1

In [ ]:
subprocess.run(
    [
        str(PYTHON_BIN),
        str(RUN_SCRIPT),
        "--setup-only",
        "--workspace-root",
        str(WORK_ROOT),
        "--stats-root",
        str(STATS_ROOT),
    ],
    cwd=REPO_ROOT,
    check=True,
    env=os.environ.copy(),
)

if not POLICY_MODEL.exists():
    raise FileNotFoundError(f"Tri-stage policy model is missing: {POLICY_MODEL}")

source_phase1 = SOURCE_WORK_ROOT / "global_checkpoints" / f"phase1_round{SOURCE_PHASE1_ROUND:03d}_global.pt"
seed_dst = WORK_ROOT / "global_checkpoints" / "round000_warmup.pt"
seed_meta = WORK_ROOT / "phase2_seed.json"
if not source_phase1.exists():
    raise FileNotFoundError(f"DQA08 phase1 seed checkpoint is missing: {source_phase1}")

expected_meta = {
    "protocol": "dqa08_2_scene_phase2_head_protected_policy_v1",
    "seed_kind": "dqa08_phase1_global_as_round000_warmup",
    "source_work_root": str(SOURCE_WORK_ROOT.resolve()),
    "source_phase1_round": SOURCE_PHASE1_ROUND,
    "source_checkpoint": str(source_phase1.resolve()),
    "target_checkpoint": str(seed_dst.resolve()),
    "client_train_scope": CLIENT_TRAIN_SCOPE,
    "uncertain_ignore": UNCERTAIN_IGNORE,
}

if seed_dst.exists() and not FORCE_SEED_PHASE1:
    if seed_meta.exists():
        current_meta = json.loads(seed_meta.read_text(encoding="utf-8"))
    else:
        current_meta = {}
    comparable = {k: current_meta.get(k) for k in expected_meta}
    if comparable != expected_meta:
        raise RuntimeError(
            "A different 08_2 seed already exists. Set FORCE_SEED_PHASE1=True only if you want to overwrite it.\n"
            f"existing={comparable}\nexpected={expected_meta}"
        )
    print("Seed already present:", seed_dst)
else:
    seed_dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source_phase1, seed_dst)
    seed_meta.write_text(json.dumps(expected_meta, indent=2), encoding="utf-8")
    print("Seeded phase2 from:", source_phase1)
    print("Seed checkpoint:", seed_dst)

manifest = json.loads((WORK_ROOT / "manifest.json").read_text(encoding="utf-8"))
server = manifest["server"]
clients = manifest["clients"]
eval_splits = manifest["paper_evaluation"]["splits"]

display(pd.DataFrame([server]))
display(pd.DataFrame(clients))
display(pd.DataFrame(eval_splits)[["name", "raw_scene", "images", "boxes"]])


## 4. Command Preview

In [ ]:
base_train_cmd = [
    str(PYTHON_BIN),
    "-u",
    str(RUN_SCRIPT),
    "--workspace-root",
    str(WORK_ROOT),
    "--stats-root",
    str(STATS_ROOT),
    "--warmup-epochs",
    str(WARMUP_EPOCHS),
    "--phase1-rounds",
    str(PHASE1_ROUNDS),
    "--phase2-rounds",
    str(PHASE2_ROUNDS),
    "--dqa-start-phase",
    str(DQA_START_PHASE),
    "--batch-size",
    str(BATCH_SIZE),
    "--workers",
    str(WORKERS),
    "--gpus",
    str(GPUS),
    "--master-port",
    str(MASTER_PORT),
    "--min-free-gib",
    str(MIN_FREE_GIB),
    "--log-file",
    str(TRAIN_LOG),
    "--classwise-blend",
    "0.35",
    "--server-anchor",
    "1.25",
    "--localize-bn",
    "--enable-dqa-guard",
    "--dqa-drop-ratio-threshold",
    "0.15",
    "--dqa-spike-ratio-threshold",
    "3.0",
]
if APPEND_TRAIN_LOG:
    base_train_cmd.append("--append-train-log")
if STREAM_TRAIN_OUTPUT:
    base_train_cmd.append("--stream-train-output")

print("08_2 skips dry-run because the shared base dry-run stops after warmup.")
print("Phase2 seed:", WORK_ROOT / "global_checkpoints" / "round000_warmup.pt")
print(" ".join(base_train_cmd))


## 5. Start or Resume Training

In [ ]:
def read_pid(path: Path) -> int | None:
    if not path.exists():
        return None
    try:
        return int(path.read_text(encoding="utf-8").strip())
    except ValueError:
        return None


def pid_state(pid: int | None) -> str:
    if pid is None:
        return "missing"
    result = subprocess.run(["ps", "-o", "stat=", "-p", str(pid)], capture_output=True, text=True)
    state = result.stdout.strip()
    if result.returncode != 0 or not state:
        return "missing"
    if "Z" in state:
        return "zombie"
    return state


train_cmd = list(base_train_cmd)
current_pid = read_pid(PID_PATH)
state = pid_state(current_pid)
print("existing pid:", current_pid, state)
print(" ".join(train_cmd))

if RUN_TRAINING and state not in {"missing", "zombie"}:
    print("Training already appears to be running.")
elif RUN_TRAINING and RUN_IN_BACKGROUND:
    env = os.environ.copy()
    RUNNER_LOG.parent.mkdir(parents=True, exist_ok=True)
    log_mode = "ab" if APPEND_TRAIN_LOG else "wb"
    with RUNNER_LOG.open(log_mode) as out:
        process = subprocess.Popen(
            train_cmd,
            cwd=REPO_ROOT,
            stdout=out,
            stderr=subprocess.STDOUT,
            env=env,
            start_new_session=True,
        )
    PID_PATH.write_text(str(process.pid), encoding="utf-8")
    print("Started PID:", process.pid)
    print("Runner log:", RUNNER_LOG)
    print("Train log:", TRAIN_LOG)
elif RUN_TRAINING:
    env = os.environ.copy()
    RUNNER_LOG.parent.mkdir(parents=True, exist_ok=True)
    log_mode = "a" if APPEND_TRAIN_LOG else "w"
    print("Running in foreground; progress will stream in this cell.")
    print("Runner log:", RUNNER_LOG)
    print("Train log:", TRAIN_LOG)
    with RUNNER_LOG.open(log_mode, encoding="utf-8", buffering=1) as runner_log:
        process = subprocess.Popen(
            train_cmd,
            cwd=REPO_ROOT,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=env,
        )
        PID_PATH.write_text(str(process.pid), encoding="utf-8")
        print("Started PID:", process.pid)
        runner_log.write(f"Started PID: {process.pid}\n")
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            runner_log.write(line)
        return_code = process.wait()
    if PID_PATH.exists() and PID_PATH.read_text(encoding="utf-8").strip() == str(process.pid):
        PID_PATH.unlink()
    if return_code != 0:
        raise RuntimeError(f"Training failed with exit code {return_code}. See {RUNNER_LOG} and {TRAIN_LOG}.")
    print("Training completed.")
else:
    print("RUN_TRAINING=False, command was not launched.")


## 6. Status

In [ ]:
def tail_lines(path: Path, lines: int = 30) -> list[str]:
    if not path.exists():
        return []
    try:
        result = subprocess.run(["tail", "-n", str(lines), str(path)], capture_output=True, text=True, check=True)
        return result.stdout.splitlines()
    except Exception:
        with path.open(encoding="utf-8", errors="replace") as f:
            return [line.rstrip("\n") for line in deque(f, maxlen=lines)]


history_path = WORK_ROOT / "history.json"
history = json.loads(history_path.read_text(encoding="utf-8")) if history_path.exists() else []
pid = read_pid(PID_PATH)

completed_phase1 = sum(1 for row in history if int(row.get("phase", 0)) == 1)
completed_phase2 = sum(1 for row in history if int(row.get("phase", 0)) == 2)
latest_global = Path(history[-1]["global"]) if history else WORK_ROOT / "global_checkpoints" / "round000_warmup.pt"

display(
    pd.DataFrame(
        [
            {
                "pid": pid,
                "pid_state": pid_state(pid),
                "completed_phase1": f"{completed_phase1}/{PHASE1_ROUNDS}",
                "completed_phase2": f"{completed_phase2}/{PHASE2_ROUNDS}",
                "completed_total": f"{len(history)}/{PHASE1_ROUNDS + PHASE2_ROUNDS}",
                "latest_global": str(latest_global),
                "seed_phase1_round": SOURCE_PHASE1_ROUND,
                "free_gib": round(shutil.disk_usage(WORK_ROOT).free / 1024**3, 2),
            }
        ]
    )
)

print("Runner log tail:")
for line in tail_lines(RUNNER_LOG, 35):
    print(line)
print("\nTrain log tail:")
for line in tail_lines(TRAIN_LOG, 35):
    print(line)


## 7. Scene/Class Evaluation

In [ ]:
RUN_EVAL = False
EVAL_SPLITS = "highway,citystreet,residential,total"
EVAL_BATCH_SIZE = 16
EVAL_DEVICE = ""

history = json.loads((WORK_ROOT / "history.json").read_text(encoding="utf-8")) if (WORK_ROOT / "history.json").exists() else []
checkpoints: list[tuple[str, Path]] = []
seed_ckpt = WORK_ROOT / "global_checkpoints" / "round000_warmup.pt"
if seed_ckpt.exists():
    checkpoints.append((f"dqa08_phase1_seed_r{SOURCE_PHASE1_ROUND:03d}", seed_ckpt))

baseline_phase2 = SOURCE_WORK_ROOT / "global_checkpoints" / f"phase2_round{SOURCE_PHASE2_BASELINE_ROUND:03d}_global.pt"
if baseline_phase2.exists():
    checkpoints.append((f"dqa08_phase2_r{SOURCE_PHASE2_BASELINE_ROUND:03d}", baseline_phase2))

phase2 = [row for row in history if int(row.get("phase", 0)) == 2]
if phase2:
    checkpoints.append((f"dqa08_2_phase2_r{int(phase2[-1]['round']):03d}", Path(phase2[-1]["global"])))

eval_cmd = [
    str(PYTHON_BIN),
    str(EVAL_SCRIPT),
    "--workspace",
    str(WORK_ROOT),
    "--splits",
    EVAL_SPLITS,
    "--batch-size",
    str(EVAL_BATCH_SIZE),
    "--no-plots",
    "--verbose",
]
if EVAL_DEVICE:
    eval_cmd.extend(["--device", EVAL_DEVICE])
for label, path in checkpoints:
    eval_cmd.extend(["--checkpoint", f"{label}={path}"])

print("checkpoints:", checkpoints)
print(" ".join(eval_cmd))
if RUN_EVAL and checkpoints:
    subprocess.run(eval_cmd, cwd=REPO_ROOT, check=True)
elif RUN_EVAL:
    print("No checkpoints found to evaluate.")
else:
    print("RUN_EVAL=False; set True after training finishes.")


## 8. Read Evaluation Tables

In [ ]:
summary_csv = WORK_ROOT / "validation_reports" / "paper_protocol_eval_summary.csv"
classwise_csv = WORK_ROOT / "validation_reports" / "paper_protocol_classwise_summary.csv"

if summary_csv.exists():
    summary = pd.read_csv(summary_csv)
    display(summary.sort_values(["checkpoint_label", "split"]))
    total = summary[summary["split"].eq("total")].copy()
    if not total.empty:
        display(total.sort_values("map50", ascending=False))
else:
    print("No summary yet:", summary_csv)

if classwise_csv.exists():
    classwise = pd.read_csv(classwise_csv)
    display(classwise.sort_values(["split", "class", "map50_95"], ascending=[True, True, False]).head(80))
    pivot = classwise.pivot_table(
        index=["split", "class"],
        columns="checkpoint_label",
        values="map50_95",
        aggfunc="first",
    )
    display(pivot)
else:
    print("No classwise summary yet:", classwise_csv)


## 9. DQA Stats Snapshot

In [ ]:
state_path = WORK_ROOT / "dqa_cwa_state.json"
if state_path.exists():
    state = json.loads(state_path.read_text(encoding="utf-8"))
    guard = state.get("round_guard", {})
    display(pd.DataFrame(guard.get("history", [])).tail(20))
    alpha = state.get("alpha", {})
    if alpha:
        latest_key = sorted(alpha)[-1]
        alpha_df = pd.DataFrame(alpha[latest_key]).T
        alpha_df.columns = latest_key.split("|")
        alpha_df.insert(0, "class", manifest["classes"])
        display(alpha_df)
else:
    print("No DQA state yet:", state_path)

stats_files = sorted(STATS_ROOT.glob("phase*_round*.json"))
print("stats files:", len(stats_files), "root:", STATS_ROOT)
if stats_files:
    latest_stats = json.loads(stats_files[-1].read_text(encoding="utf-8"))
    rows = latest_stats.get("class_stats", latest_stats.get("classes", []))
    if isinstance(rows, list) and rows:
        display(pd.DataFrame(rows).head(20))
    else:
        print("latest stats:", stats_files[-1])


## 10. Head-Protected Policy Schedule

In [ ]:
if THRESHOLD_LOG.exists():
    records = [json.loads(line) for line in THRESHOLD_LOG.read_text(encoding="utf-8").splitlines() if line.strip()]
    gate_df = pd.DataFrame(records)
    display(gate_df.tail(30))
    if not gate_df.empty:
        compact = gate_df[["phase", "round", "client_id", "enabled", "reason", "nms_conf_thres", "teacher_loss_weight", "source_stats"]].copy()
        compact["low_min"] = gate_df["ignore_thres_low"].map(lambda xs: min(xs) if isinstance(xs, list) else None)
        compact["low_max"] = gate_df["ignore_thres_low"].map(lambda xs: max(xs) if isinstance(xs, list) else None)
        compact["mid_min"] = gate_df["ignore_thres_mid"].map(lambda xs: min(xs) if isinstance(xs, list) else None)
        compact["mid_max"] = gate_df["ignore_thres_mid"].map(lambda xs: max(xs) if isinstance(xs, list) else None)
        compact["high_min"] = gate_df["ignore_thres_high"].map(lambda xs: min(xs) if isinstance(xs, list) else None)
        compact["high_max"] = gate_df["ignore_thres_high"].map(lambda xs: max(xs) if isinstance(xs, list) else None)
        display(compact.tail(40))
else:
    print("No head-protected policy log yet:", THRESHOLD_LOG)
